# Subgraphs and Nested Workflows

Large LangGraph applications outgrow a single flat diagram. **Subgraphs** let you **compile a graph once** and **embed it as a node** in a parent graph — encapsulating a ReAct loop, RAG pipeline, or approval flow behind a single boundary.

This notebook covers **`builder.add_node("rag", rag_subgraph)`**, parent/child **state schemas** (shared keys vs transforms), nesting a **ReAct agent** inside the **05** router graph, compiling subgraphs separately, **Mermaid** nested visualization, when to use subgraphs vs flat nodes, and a multi-stage **classify → doc_agent | math_agent** pipeline.

Prerequisites: **05. Building Your First Graph**, **06. Agent Loops and the ReAct Pattern**, **04. Conditional Routing and Branching**.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import json
import operator
from typing import Annotated, Literal
from typing_extensions import TypedDict

import langchain_core

try:
    import langgraph
    print("langgraph:", langgraph.__version__)
except ImportError:
    print("langgraph: pip install langgraph")

print("langchain-core:", langchain_core.__version__)

---

## 1. What Is a Subgraph?

A **subgraph** is a **`CompiledStateGraph`** passed directly to **`add_node`**. The parent graph treats it like any other node — but internally the subgraph runs its own nodes, edges, and routers.

```
Parent graph:
  START → classify ──┬── docs ──► ┌─────────────┐ ──► END
                     │            │  doc_agent  │
                     │            │  (subgraph) │
                     │            └─────────────┘
                     └── math ──► ┌─────────────┐ ──► END
                                  │ math_agent  │
                                  │ (subgraph)  │
                                  └─────────────┘
```

| Concept | Meaning |
|---------|--------|
| **Parent graph** | Top-level router / orchestrator |
| **Child subgraph** | Self-contained workflow compiled separately |
| **Embedding** | `add_node("name", compiled_child)` |
| **State handoff** | Shared keys pass through; extra keys stay local |

Notebook **06** noted wrapping the ReAct loop in a larger graph — subgraphs make that **explicit and reusable**.

---

## 2. Subgraph = Compiled Graph as Node

The pattern is always: **build → compile → embed**.

In [ ]:
from langgraph.graph import END, START, StateGraph


class MiniState(TypedDict):
    value: int
    trace: Annotated[list[str], operator.add]


def double_node(state: MiniState) -> dict:
    return {"value": state["value"] * 2, "trace": ["double"]}


def add_ten_node(state: MiniState) -> dict:
    return {"value": state["value"] + 10, "trace": ["add_ten"]}


# --- Child subgraph: double → add_ten ---
child_builder = StateGraph(MiniState)
child_builder.add_node("double", double_node)
child_builder.add_node("add_ten", add_ten_node)
child_builder.add_edge(START, "double")
child_builder.add_edge("double", "add_ten")
child_builder.add_edge("add_ten", END)
child_subgraph = child_builder.compile()

# --- Parent graph: wrap subgraph as one node ---
def prep_node(state: MiniState) -> dict:
    return {"value": state["value"] + 1, "trace": ["prep"]}


parent_builder = StateGraph(MiniState)
parent_builder.add_node("prep", prep_node)
parent_builder.add_node("child_pipeline", child_subgraph)  # compiled graph as node
parent_builder.add_edge(START, "prep")
parent_builder.add_edge("prep", "child_pipeline")
parent_builder.add_edge("child_pipeline", END)

parent_graph = parent_builder.compile()
result = parent_graph.invoke({"value": 3, "trace": []})
print("value:", result["value"], "(expect (3+1)*2+10 = 18)")
print("trace:", result["trace"])

**`trace`** shows parent and child nodes: `prep → double → add_ten`. The parent sees one edge to **`child_pipeline`**, but the subgraph runs multiple internal steps.

---

## 3. Parent/Child State Schemas

LangGraph subgraphs work best when parent and child share **compatible state keys**:

| Strategy | When | How |
|----------|------|-----|
| **Shared schema** | Parent and child need same fields | One `TypedDict`; both graphs use it |
| **Shared keys, extra child keys** | Child has internal-only fields | Overlapping keys pass through; child ignores parent extras |
| **Different schemas** | Parent state shape differs | Wrapper node: `child.invoke(subset)` + map back |

```
ParentState:  { question, intent, answer, trace }
ChildState:   { question, context, answer, trace }   ← shared keys auto-map

Wrapper pattern (different schemas):
  parent → wrapper_node → child.invoke({messages: ...}) → patch parent state
```

Prefer **shared keys** for `messages`, `question`, `answer`, and `trace` across agent subgraphs.

---

## 4. Teaching Corpus — Notes API (n1–n4, c1–c5)

Same corpus as **05** and **06**:

In [ ]:
CORPUS = [
    {"id": "c1", "text": "The Notes API is built with FastAPI. Routes live under /notes with GET, POST, PUT, DELETE."},
    {"id": "c2", "text": "Alembic applies SQLAlchemy schema migrations. Run alembic upgrade head after pulling revisions."},
    {"id": "c3", "text": "JWT bearer tokens authenticate API requests. Send Authorization: Bearer token header."},
    {"id": "c4", "text": "Pytest fixtures share database setup. Use conftest.py for session-scoped engines."},
    {"id": "c5", "text": "Alembic autogenerate compares SQLAlchemy models to the live schema and drafts revision files."},
]

NOTES = {
    "n1": "The Notes API is built with FastAPI. Routes live under /notes.",
    "n2": "Run alembic upgrade head after pulling database migrations.",
    "n3": "JWT bearer tokens use Authorization: Bearer header.",
    "n4": "Pytest fixtures for DB setup belong in conftest.py.",
}


def keyword_retrieve(query: str, k: int = 2) -> tuple[str, list[str]]:
    tokens = set(query.lower().split())
    scored = [(len(tokens & set(d["text"].lower().split())), d) for d in CORPUS]
    scored.sort(key=lambda x: x[0], reverse=True)
    top = [d for s, d in scored if s > 0][:k] or [CORPUS[0]]
    context = "\n".join(f"[{d['id']}] {d['text']}" for d in top)
    return context, [d["id"] for d in top]

---

## 5. Build the Doc Agent Subgraph (RAG Path)

Extract the **retrieve → grade → generate** path from **05** as a standalone subgraph:

In [ ]:
class PipelineState(TypedDict):
    question: str
    intent: str
    context: str
    source_ids: list[str]
    grade: str
    answer: str
    trace: Annotated[list[str], operator.add]


def retrieve_node(state: PipelineState) -> dict:
    context, ids = keyword_retrieve(state["question"])
    return {"context": context, "source_ids": ids, "trace": ["retrieve"]}


def grade_node(state: PipelineState) -> dict:
    q_tokens = set(state["question"].lower().split())
    ctx_tokens = set(state["context"].lower().split())
    grade = "relevant" if len(q_tokens & ctx_tokens) >= 2 else "irrelevant"
    return {"grade": grade, "trace": ["grade"]}


def generate_node(state: PipelineState) -> dict:
    answer = f"Based on {state['source_ids']}: {state['context'][:120]}..."
    return {"answer": answer, "trace": ["generate"]}


def fallback_node(state: PipelineState) -> dict:
    return {"answer": "Could not find relevant docs.", "trace": ["fallback"]}


def route_grade(state: PipelineState) -> Literal["generate", "fallback"]:
    return "generate" if state["grade"] == "relevant" else "fallback"


def build_doc_subgraph():
    builder = StateGraph(PipelineState)
    builder.add_node("retrieve", retrieve_node)
    builder.add_node("grade", grade_node)
    builder.add_node("generate", generate_node)
    builder.add_node("fallback", fallback_node)
    builder.add_edge(START, "retrieve")
    builder.add_edge("retrieve", "grade")
    builder.add_conditional_edges("grade", route_grade)
    builder.add_edge("generate", END)
    builder.add_edge("fallback", END)
    return builder.compile()


doc_subgraph = build_doc_subgraph()
doc_out = doc_subgraph.invoke({
    "question": "JWT bearer token header?",
    "intent": "", "context": "", "source_ids": [], "grade": "", "answer": "", "trace": [],
})
print("doc trace:", doc_out["trace"])
print("answer:", doc_out["answer"][:100])

Test subgraphs **in isolation** before embedding — same discipline as unit-testing routers (**04**).

---

## 6. Build the Math Agent Subgraph (ReAct Loop)

A minimal **model ↔ tools** cycle for arithmetic — from **06**:

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph.message import add_messages


class MathAgentState(TypedDict):
    messages: Annotated[list, add_messages]
    answer: str
    trace: Annotated[list[str], operator.add]


@tool
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b


MATH_TOOLS = [add, multiply]
MATH_BY_NAME = {t.name: t for t in MATH_TOOLS}


def fake_math_model(state: MathAgentState) -> dict:
    text = state["messages"][-1].content.lower()
    if any(isinstance(m, ToolMessage) for m in state["messages"]):
        ai = AIMessage(content="The sum is 42.")
    elif "add" in text or "plus" in text:
        ai = AIMessage(
            content="",
            tool_calls=[{"name": "add", "args": {"a": 12, "b": 30}, "id": "m1", "type": "tool_call"}],
        )
    else:
        ai = AIMessage(content="Ask an addition or multiplication question.")
    return {"messages": [ai], "trace": ["math_model"]}


def math_tools(state: MathAgentState) -> dict:
    last = state["messages"][-1]
    msgs = []
    for tc in last.tool_calls:
        result = str(MATH_BY_NAME[tc["name"]].invoke(tc["args"]))
        msgs.append(ToolMessage(content=result, tool_call_id=tc["id"]))
    return {"messages": msgs, "trace": ["math_tools"]}


def math_router(state: MathAgentState):
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and last.tool_calls:
        return "math_tools"
    return END


def math_finalize(state: MathAgentState) -> dict:
    return {"answer": state["messages"][-1].content, "trace": ["math_finalize"]}


def build_math_subgraph():
    builder = StateGraph(MathAgentState)
    builder.add_node("math_model", fake_math_model)
    builder.add_node("math_tools", math_tools)
    builder.add_node("math_finalize", math_finalize)
    builder.add_edge(START, "math_model")
    builder.add_conditional_edges("math_model", math_router, {"math_tools": "math_tools", END: "math_finalize"})
    builder.add_edge("math_tools", "math_model")
    builder.add_edge("math_finalize", END)
    return builder.compile()


math_subgraph = build_math_subgraph()
math_out = math_subgraph.invoke({
    "messages": [HumanMessage(content="What is 12 plus 30?")],
    "answer": "", "trace": [],
})
print("math trace:", math_out["trace"])
print("answer:", math_out["answer"])

The math subgraph owns the **ReAct cycle** internally. The parent only needs the final **`answer`** key.

---

## 7. Unified Parent State

Merge doc and math fields into one **parent schema** so both subgraphs can embed cleanly:

In [ ]:
class OrchestratorState(TypedDict):
    question: str
    intent: str
    # doc subgraph keys
    context: str
    source_ids: list[str]
    grade: str
    # math subgraph keys
    messages: Annotated[list, add_messages]
    # shared output
    answer: str
    trace: Annotated[list[str], operator.add]


DOCS_KEYWORDS = {"jwt", "alembic", "pytest", "fastapi", "migration", "token", "route", "conftest", "doc"}
MATH_KEYWORDS = {"add", "plus", "sum", "multiply", "times", "calculate", "math"}


def classify_node(state: OrchestratorState) -> dict:
    q = state["question"].lower()
    if any(k in q for k in MATH_KEYWORDS):
        intent = "math"
    elif any(k in q for k in DOCS_KEYWORDS):
        intent = "docs"
    else:
        intent = "general"
    return {"intent": intent, "trace": ["classify"]}


def general_node(state: OrchestratorState) -> dict:
    return {
        "answer": "Ask about Notes API docs (n1-n4) or arithmetic.",
        "trace": ["general"],
    }


def route_intent(state: OrchestratorState) -> Literal["docs", "math", "general"]:
    return state["intent"]  # type: ignore[return-value]


def math_adapter(state: OrchestratorState) -> dict:
    """Seed messages for math subgraph from parent question."""
    return {"messages": [HumanMessage(content=state["question"])]}

**`math_adapter`** is a thin parent node that prepares child input when schemas differ slightly. With shared keys, subgraphs receive the full parent state automatically.

---

## 8. Compile Subgraphs, Then Embed in Parent

Factory pattern from **05** — compile each piece once, assemble the orchestrator:

In [ ]:
def build_orchestrator_graph():
    doc_sg = build_doc_subgraph()
    math_sg = build_math_subgraph()

    builder = StateGraph(OrchestratorState)
    builder.add_node("classify", classify_node)
    builder.add_node("doc_agent", doc_sg)       # subgraph as node
    builder.add_node("math_adapter", math_adapter)
    builder.add_node("math_agent", math_sg)     # subgraph as node
    builder.add_node("general", general_node)

    builder.add_edge(START, "classify")
    builder.add_conditional_edges(
        "classify",
        route_intent,
        {"docs": "doc_agent", "math": "math_adapter", "general": "general"},
    )
    builder.add_edge("math_adapter", "math_agent")
    builder.add_edge("doc_agent", END)
    builder.add_edge("math_agent", END)
    builder.add_edge("general", END)
    return builder.compile()


orchestrator = build_orchestrator_graph()
INITIAL = {
    "question": "", "intent": "", "context": "", "source_ids": [],
    "grade": "", "messages": [], "answer": "", "trace": [],
}

Note the path map: router returns **`docs`** / **`math`** but node names are **`doc_agent`** / **`math_agent`** — same pattern as **04** and **05**.

---

## 9. Multi-Stage Pipeline — Classify → Subgraph

Run questions through the orchestrator and inspect traces:

In [ ]:
test_questions = [
    ("JWT bearer token header format?", "docs"),
    ("What is 12 plus 30?", "math"),
    ("Hello there", "general"),
]

for question, expected in test_questions:
    out = orchestrator.invoke({**INITIAL, "question": question})
    print(f"Q: {question}")
    print(f"  intent={out['intent']} (expect {expected})")
    print(f"  trace={out['trace']}")
    print(f"  answer={out['answer'][:90]}...\n")

Doc questions trace through **`retrieve → grade → generate`** inside **`doc_agent`**. Math questions loop through **`math_model ↔ math_tools`** inside **`math_agent`**.

---

## 10. Mermaid — Nested Structure

Export the parent graph — subgraph internals appear as nested clusters:

In [ ]:
try:
    mermaid = orchestrator.get_graph(xray=True).draw_mermaid()
    print(mermaid[:1200], "..." if len(mermaid) > 1200 else "")
except TypeError:
    # older langgraph — xray may be unavailable
    mermaid = orchestrator.get_graph().draw_mermaid()
    print(mermaid[:1200], "..." if len(mermaid) > 1200 else "")
except Exception as e:
    print("Mermaid export:", e)

Paste into [mermaid.live](https://mermaid.live). **`xray=True`** expands subgraph contents — essential for architecture reviews.

```mermaid
flowchart TD
    START --> classify
    classify -->|docs| doc_agent
    classify -->|math| math_adapter
    classify -->|general| general
    math_adapter --> math_agent
    subgraph doc_agent [doc_agent subgraph]
        retrieve --> grade --> generate
    end
    subgraph math_agent [math_agent subgraph]
        math_model --> math_tools --> math_model
    end
```

---

## 11. Subgraph vs Flat Nodes

When to extract a subgraph vs keeping nodes in the parent:

| Use **subgraph** | Use **flat nodes** |
|------------------|-------------------|
| Reusable across products (doc agent in 3 apps) | One-off glue logic |
| Team owns a bounded module | Single developer prototype |
| Independent test + deploy cycle | < 5 nodes, no reuse |
| Different checkpointer / interrupt policy per module | Uniform HITL policy (**09**) |
| Hide internal complexity from parent diagram | Parent needs to route inside child |

**Rule of thumb:** if you would import it as `from agents.doc import build_doc_graph`, make it a subgraph. If it is a single `format_answer` helper, keep it flat.

---

## 12. State Transform Wrapper Pattern

When parent and child schemas diverge, wrap the compiled subgraph in a plain node:

In [ ]:
class ParentOnlyState(TypedDict):
    question: str
    answer: str


def doc_wrapper_node(state: ParentOnlyState) -> dict:
    child_in = {
        "question": state["question"], "intent": "docs",
        "context": "", "source_ids": [], "grade": "", "answer": "", "trace": [],
    }
    child_out = doc_subgraph.invoke(child_in)
    return {"answer": child_out["answer"]}


wrapper_builder = StateGraph(ParentOnlyState)
wrapper_builder.add_node("doc_wrapper", doc_wrapper_node)
wrapper_builder.add_edge(START, "doc_wrapper")
wrapper_builder.add_edge("doc_wrapper", END)
wrapper_graph = wrapper_builder.compile()

wrap_out = wrapper_graph.invoke({"question": "Alembic upgrade command?", "answer": ""})
print("wrapper answer:", wrap_out["answer"][:100])

Use wrappers when integrating third-party graphs you cannot change, or when the parent should not see child-internal keys like **`grade`** or **`messages`**.

---

## 13. Streaming Through Subgraphs

Parent-level streaming emits updates for subgraph internal nodes (**13**):

In [ ]:
print("Streaming doc path:")
for update in orchestrator.stream(
    {**INITIAL, "question": "pytest conftest fixtures?"},
    stream_mode="updates",
):
    node = list(update.keys())[0]
    print(f"  {node}: trace+={update[node].get('trace', [])}")

Subgraph node names (`retrieve`, `grade`, `generate`) surface in stream events — useful for progress bars without exposing the parent's routing logic to the UI.

---

## 14. Checkpointing and HITL at Subgraph Boundaries

Apply interrupts on the **parent** or **child** compile call (**08**, **09**):

| Policy | Where to compile | Example |
|--------|------------------|--------|
| Pause before any doc answer | Parent | `interrupt_before=["doc_agent"]` |
| Pause before tool execution | Child math subgraph | `build_math_subgraph(checkpointer=...)` |
| Single audit thread | Parent checkpointer only | Child runs ephemeral |

```python
# Parent-level HITL on subgraph entry
orchestrator = builder.compile(
    checkpointer=MemorySaver(),
    interrupt_before=["doc_agent", "math_agent"],
)
```

Prefer **one checkpointer on the parent** unless child subgraphs are also invoked standalone.

---

## 15. Design Guidelines

| Guideline | Rationale |
|-----------|----------|
| **Compile subgraphs in factories** | `build_doc_subgraph()` — testable, reusable |
| **Shared output keys** | Parent reads `answer` from any child |
| **Keep parent router thin** | Classify in one node; delegate to subgraphs |
| **Test subgraphs alone first** | Isolate bugs before orchestration |
| **`xray` Mermaid for docs** | Onboarding and PR reviews |
| **Avoid deep nesting (>2 levels)** | Hard to debug; flatten or use supervisor (**11**) |
| **Match LangChain track** | LCEL RAG chain lives inside doc subgraph's `generate` node |

---

## 16. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Pass `StateGraph` builder instead of compiled graph | Type error at compile | `.compile()` first |
| Mismatched state keys | Child sees empty fields | Align schemas or use wrapper |
| Router label ≠ node name | Wrong branch | Path map (**04**) |
| Rebuild subgraph per request | Slow | Compile at startup (**16**) |
| Nested subgraphs without `xray` | Opaque diagrams | `get_graph(xray=True)` |
| Duplicate node names across subgraphs | Confusing traces | Prefix child nodes (`doc_retrieve`) |
| Child modifies keys parent does not expect | Polluted parent state | Restrict child schema |
| Two checkpointers on parent+child | Conflicting thread state | One checkpointer on parent |

---

## 17. Summary

A **subgraph** is a **`CompiledStateGraph`** embedded with **`add_node("name", compiled_graph)`**. Parent and child share overlapping **state keys**; wrapper nodes bridge incompatible schemas. Compile subgraphs **separately**, test in isolation, then wire into a **classify → subgraph** orchestrator.

Key takeaways:

- **`doc_agent`** encapsulates the **05** RAG path; **`math_agent`** encapsulates the **06** ReAct loop.
- **Path maps** connect router labels to subgraph node names.
- **Mermaid `xray`** reveals nested structure for documentation.
- **Subgraphs** beat flat nodes when modules are **reusable** and **team-owned**.
- **Checkpointing / HITL** usually belongs on the **parent** graph (**08**, **09**).

Demonstrations built mini parent/child graphs, doc and math subgraphs, a full orchestrator with classify routing, wrapper transforms, streaming, and Mermaid export.

Next: **11. Multi-Agent and Supervisor Patterns** — coordinate multiple subgraphs with a supervisor router.